# Dynamic Limit Engine — Live Data Exploration

Runs `compute_limit` on every commercial connection in `payouts.csv`.  
All payouts are treated as provisional (non-routed). Capture score is always the 0.50 floor
since there is no routed flow, so revenue is not sent.

**Fixed defaults for this run:**
- `merchant_score` = 0.60 — both PBS and rating unknown, each defaults to 0.6, `min(0.6, 0.6)`
- `capture_score` = 0.50 — floor, no routed flow
- `l` (legal_security_norm) = 0.50 — no instruments signed
- `v` (verification_norm) = 1.0 — shopify_payments is tier 1
- `j` (jurisdiction) = 1.0 GB · 0.70 SE · 0.60 DK/FI/unknown
- `flow_score` = 1.0 — no routed months, gate never triggers

In [2]:
import warnings
warnings.filterwarnings("ignore")

from datetime import date

import pandas as pd

from app.engine import compute_limit
from app.models import ChannelInput, MerchantLimitRequest, Payout

## 1 · Load data

In [3]:
payouts_raw = pd.read_csv("payouts.csv")
payouts_raw["date"] = pd.to_datetime(payouts_raw["Issued At"]).dt.date
payouts_raw = payouts_raw[payouts_raw["Status"] == "PAID"].copy()

print(f"Payouts: {len(payouts_raw):,} rows · {payouts_raw['Commercial Connection ID'].nunique()} connections")

Payouts: 4,423 rows · 16 connections


## 2 · Config

In [4]:
AS_OF = date(2026, 6, 26)  # last fully-settled date in the data
REVENUE_CCY = "GBP"

## 4 · Build requests

In [5]:
def build_payouts(group: pd.DataFrame) -> list[Payout]:
    return [
        Payout(date=r.date, amount=r.Amount, currency=r.Currency, routed_to_treyd=False, encumbered=False)
        for r in group.itertuples()
    ]


requests: dict[str, dict] = {}

for conn_id, conn_group in payouts_raw.groupby("Commercial Connection ID"):
    company = conn_group["Core Company - Company \u2192 Name"].iloc[0]
    country = conn_group["Core Company - Company \u2192 Country"].iloc[0]

    requests[conn_id] = {
        "company": company,
        "request": MerchantLimitRequest(
            merchant_id=conn_id,
            revenue_currency=REVENUE_CCY,
            country=country,
            channels=[ChannelInput(
                channel_id=conn_id,
                channel_type="shopify_payments",
                payouts=build_payouts(conn_group),
            )],
        ),
    }

print(f"Built {len(requests)} requests")

Built 16 requests


## 5 · Run engine

In [6]:
results: dict[str, dict] = {}

for conn_id, info in requests.items():
    resp = compute_limit(info["request"], AS_OF)
    results[conn_id] = {"company": info["company"], "response": resp}

print(f"Computed limits for {len(results)} connections")

Computed limits for 16 connections


## 3 · Summary — one row per connection × currency

In [9]:
rows = []
for conn_id, info in results.items():
    mt = info["response"].merchant_trace
    for lim in info["response"].limits:
        ch = lim.channels[0]
        rows.append({
            "commercial_connection_id": conn_id,
            "company": info["company"],
            "country": requests[conn_id]["request"].country,
            "currency": lim.currency,
            "flow_base": ch.flow_base,
            "v": ch.verification_norm,
            "l": ch.legal_security_norm,
            "j": ch.jurisdiction,
            "flow_score": ch.flow_score,
            "q": ch.quality_q,
            "base_months": mt.base_months,
            "capture_score": mt.capture_score,
            "merchant_score": mt.merchant_score,
            "merchant_multiplier": round(mt.base_months * mt.capture_score * mt.merchant_score, 4),
            "provisional_limit": lim.dynamic_limit,
        })

summary = (
    pd.DataFrame(rows)
    .sort_values("provisional_limit", ascending=False)
    .reset_index(drop=True)
)

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 220)

summary

,commercial_connection_id,company,currency,base_months,merchant_score,merchant_multiplier,flow_base,j,l,flow_score,q,provisional_limit
0,01KV6DZZGJ2DJT6N1W3PF0ZMV8,Jeanerica AB,SEK,2.00,0.60,0.72,"1,737,059.02",0.70,0.50,1.00,0.35,"437,738.87"
1,01KV6DZZGJ2DJT6N1W3PF0ZMV8,Jeanerica AB,NOK,2.00,0.60,0.72,"697,319.29",0.70,0.50,1.00,0.35,"175,724.46"
2,01KVWRPC1H4Q09H57K0636XP58,Local Rule AB,SEK,3.00,0.60,1.08,"394,188.61",0.70,0.50,1.00,0.35,"149,003.29"
3,01KV88169WS61VY7KKY0F74Q9K,Latalia AB,SEK,3.00,0.60,1.08,"256,782.19",0.70,0.50,1.00,0.35,"97,063.67"
4,01KV81WPHSGEP6EZ6FRSJV40Y3,Alobha Label AB,SEK,3.00,0.60,1.08,"231,983.13",0.70,0.50,1.00,0.35,"87,689.62"
5,01KVFHFTDM4404P35TWV4CNF7M,SIGMA ADVANTAGE LIMITED,GBP,3.00,0.60,1.08,"152,759.50",1.00,0.50,1.00,0.50,"82,490.13"
6,01KV5ZAV82H7TP1D2CHDZCGYFX,FLICO LTD,GBP,3.00,0.60,0.90,"113,282.83",1.00,0.50,1.00,0.50,"50,977.27"
7,01KV5CY7K74XKX0EGEYFH6QVJF,LA CASA DE JACK LTD,GBP,3.00,0.60,0.90,"85,942.52",1.00,0.50,1.00,0.50,"38,674.13"
8,01KTXSTYQQRTMPJTG8V8D5Q8YF,BRAINGAIN LIMITED,EUR,2.00,0.60,0.72,"99,543.69",1.00,0.50,1.00,0.50,"35,835.73"
9,01KV5X2SJSQR6TVQGF0GZR6WZF,EXTRACTED LTD,GBP,3.00,0.60,0.90,"73,768.45",1.00,0.50,1.00,0.50,"33,195.80"


## 4 · SQL-ready VALUES clause

In [10]:
sql_cols = [
    "commercial_connection_id", "company", "country", "currency",
    "flow_base",
    "v", "l", "j", "flow_score", "q",
    "base_months", "capture_score", "merchant_score", "merchant_multiplier",
    "provisional_limit",
]

lines = []
for _, r in summary[sql_cols].iterrows():
    vals = ", ".join(
        f"'{v}'" if isinstance(v, str) else f"{v:.4f}" if isinstance(v, float) else str(v)
        for v in r
    )
    lines.append(f"  ({vals})")

col_list = ", ".join(sql_cols)
print(f"-- dynamic_limit engine output as_of {AS_OF}")
print(f"-- provisional flow only, all defaults (no PBS/rating/instruments/routing)")
print(f"WITH engine_limits ({col_list}) AS (VALUES")
print(",\n".join(lines))
print(")")

-- dynamic_limit engine output as_of 2026-06-26
-- provisional flow only, all defaults (no PBS/rating/instruments/routing)
WITH engine_limits (commercial_connection_id, company, currency, base_months, merchant_score, merchant_multiplier, flow_base, j, l, flow_score, q, provisional_limit) AS (VALUES
  ('01KV6DZZGJ2DJT6N1W3PF0ZMV8', 'Jeanerica AB', 'SEK', 2.0000, 0.6000, 0.7200, 1737059.0200, 0.7000, 0.5000, 1.0000, 0.3500, 437738.8700),
  ('01KV6DZZGJ2DJT6N1W3PF0ZMV8', 'Jeanerica AB', 'NOK', 2.0000, 0.6000, 0.7200, 697319.2900, 0.7000, 0.5000, 1.0000, 0.3500, 175724.4600),
  ('01KVWRPC1H4Q09H57K0636XP58', 'Local Rule AB', 'SEK', 3.0000, 0.6000, 1.0800, 394188.6100, 0.7000, 0.5000, 1.0000, 0.3500, 149003.2900),
  ('01KV88169WS61VY7KKY0F74Q9K', 'Latalia AB', 'SEK', 3.0000, 0.6000, 1.0800, 256782.1900, 0.7000, 0.5000, 1.0000, 0.3500, 97063.6700),
  ('01KV81WPHSGEP6EZ6FRSJV40Y3', 'Alobha Label AB', 'SEK', 3.0000, 0.6000, 1.0800, 231983.1300, 0.7000, 0.5000, 1.0000, 0.3500, 87689.6200),
  ('